# R3_06 — Evaluasi (SATU KALI)

Kode statistik sama dengan `revision/01`, `02`, `06` dan NB07 (fungsi di `r3_eval.py` memakai `revision/revision_utils.py`):
ACD/ASC/Overall F1, akurasi, per kelas, CI cluster bootstrap per tweet (B = 10.000, seed 42), uji permutasi berpasangan tingkat cluster,
McNemar (metode naskah) + McNemar berkluster, koreksi Holm atas 15 perbandingan, analisis kasus yang ditandai, transisi label,
perbandingan LLM-only/LLM-always, panggilan/token/waktu API, ablasi V1–V6 (Tabel 16), skema Tabel 17 terpilih, audit silver konfigurasi
baru (Tabel 6), disagreement, Gambar 2–4, dan `perbandingan_lama_vs_baru.md`.

* **`DRY_RUN = True` (default):** memakai keluaran dry (label **dev**) ke `outputs_round3/dry_run/evaluation/` — untuk uji kode.
* **Mode nyata:** memuat label uji. Berhenti bila kode/data/checkpoint tidak cocok dengan `frozen_choices.json` atau evaluasi sudah pernah
  dijalankan (`FORCE_RERUN=True` hanya untuk memperbaiki galat implementasi; dicatat di `RERUN_LOG.json`).
* Penomoran tabel naskah ada di `pemetaan_tabel_naskah.md` (perlu dikonfirmasi peneliti).

In [ ]:
# ── PARAMETER ──
DRY_RUN = True
FORCE_RERUN = False
RERUN_REASON = ''              # wajib diisi bila FORCE_RERUN=True

import os
DRY_RUN = os.environ.get('R3_DRY_RUN', str(DRY_RUN)) == 'True'

In [ ]:
import sys, json, itertools, warnings
sys.path.insert(0, os.environ.get('R3_CODE_DIR', os.getcwd()))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import config as C
import r3_lib as L
import r3_eval as E
ru = E.ru
warnings.filterwarnings('ignore')
D = L.mode_dirs(DRY_RUN)
if DRY_RUN:
    L.forbid_test_access(allow_dirs=[C.DRY_DIR])
frozen = L.verify_frozen(strict=not DRY_RUN, check_models=False)
OUT = C.assert_writable(D.eval)
for need in [D.test_pred / 'DONE.json', D.llm / 'DONE.json']:
    assert need.exists(), f'Belum ada: {need}'
if not DRY_RUN:
    assert not json.loads((D.llm / 'DONE.json').read_text())['mock_used'], 'label LLM mock tidak boleh dipakai di mode nyata'
    if (OUT / 'DONE.json').exists():
        if not FORCE_RERUN:
            raise RuntimeError('Evaluasi sudah pernah dijalankan (DONE.json). Hanya FORCE_RERUN=True + RERUN_REASON untuk galat implementasi.')
        assert RERUN_REASON.strip(), 'Isi RERUN_REASON'
        log = json.loads((OUT / 'RERUN_LOG.json').read_text()) if (OUT / 'RERUN_LOG.json').exists() else []
        L.write_json(OUT / 'RERUN_LOG.json', log + [{'at': L.now_utc(), 'reason': RERUN_REASON}])
OUT.mkdir(parents=True, exist_ok=True)
FIG = OUT / 'figures'; FIG.mkdir(exist_ok=True)
SAVE = lambda df, name, title='', index=False, ff='.4f': ru.save_table(df, OUT, name, title, floatfmt=ff, index=index)
print(f"Mode: {'DRY_RUN (label dev)' if DRY_RUN else 'NYATA (label uji)'} | output: {OUT}")

In [ ]:
df, probs = E.build_eval_frame(D, frozen)
E.check_sklearn_equivalence(df, E.SCHEMES6 + ['LLM-only', 'LLM-always'])
n, n_tw = len(df), df.tweet_id.nunique()
print(f'{n} pasangan, {n_tw} tweet; gold: ' + str(df.y_true.map(C.LABEL_MAP_INV).value_counts().to_dict()))
print(f"Ditandai: {int(df.flagged.sum())} | keputusan: {df.decision.value_counts().to_dict()}")

## Tabel utama: enam skema (ACD/ASC/Overall F1, akurasi) dengan CI 95% cluster bootstrap

In [ ]:
main, n_missing = E.metrics_ci(df, E.SCHEMES6)
SAVE(main, 'T_utama_enam_skema_ci', 'Enam skema: estimasi dan CI 95% cluster bootstrap per tweet')
SAVE(E.pretty_metrics(main), 'T_utama_enam_skema', 'Enam skema: estimasi [CI 95%]')
asp_rows = []
for s in E.SCHEMES6:
    for a in C.ASPECTS:
        sub = df[df.aspek == a]
        m = ru.sklearn_metrics(sub.y_true.values, sub[s].values)
        asp_rows.append({'skema': s, 'aspek': a, 'ACD F1': m['acd_f1'], 'ASC F1': m['asc_f1']})
SAVE(pd.DataFrame(asp_rows), 'T_per_aspek', 'ACD dan ASC F1 per aspek (NB07 §4)')
print('Resample tanpa suatu kelas gold:', n_missing)
E.pretty_metrics(main)

## Per kelas

In [ ]:
pc = E.per_class_ci(df, E.SCHEMES6); SAVE(pc, 'T_per_kelas_4kelas', 'P/R/F1 per kelas (560 pasangan), CI 95%')
apc = E.asc_per_class_ci(df, E.SCHEMES6); SAVE(apc, 'T_per_kelas_ASC', 'P/R/F1 kelas sentimen di dalam definisi ASC, CI 95%')
pc[pc.skema == E.REF].round(3)

## Tabel 15: uji signifikansi 15 perbandingan (Holm)

In [ ]:
pw_asc = E.pairwise(df, metric='asc_f1', seed_offset=1000)
pw_ovr = E.pairwise(df, metric='overall_f1', seed_offset=2000)
pw_acc = E.pairwise(df, metric='accuracy', seed_offset=3000)
mc = E.mcnemar_family(df, pw_acc=pw_acc)
for t, nm in [(pw_asc, 'asc_f1'), (pw_ovr, 'overall_f1'), (pw_acc, 'accuracy')]:
    SAVE(t, f'T_berpasangan_{nm}', f'Selisih {nm} berpasangan, CI 95% cluster bootstrap, p permutasi cluster, Holm 15 perbandingan')
SAVE(mc, 'T_mcnemar', 'McNemar metode naskah dan versi berkluster (Holm 15 perbandingan)')
t15 = E.significance_table(pw_asc, pw_ovr, mc)
SAVE(t15, 'T15_signifikansi', 'Tabel 15 — ✔ SIGNIFIKAN = p Holm < 0,05; ✘ = tidak signifikan setelah Holm')
t15

In [ ]:
# Forest plot selisih terhadap Soft Voting + LLM (revision/01 §5)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, tab, title in zip(axes, [pw_asc, pw_ovr], ['ASC F1', 'Overall F1']):
    t = tab[tab.skema_A == E.REF].iloc[::-1]; y = np.arange(len(t)); sig = t.signifikan_holm_005.values
    ax.errorbar(t.delta_observasi, y, xerr=[t.delta_observasi - t.ci_lo, t.ci_hi - t.delta_observasi], fmt='none', ecolor='#555', capsize=3)
    ax.scatter(t.delta_observasi[sig], y[sig], color='#1f5fa8', zorder=3, label='p Holm < 0,05')
    ax.scatter(t.delta_observasi[~sig], y[~sig], facecolor='white', edgecolor='#1f5fa8', zorder=3, label='p Holm ≥ 0,05')
    ax.axvline(0, color='#999', ls='--'); ax.set_yticks(y); ax.set_yticklabels(t.skema_B)
    ax.set_xlabel(f'Δ {title} (Soft Voting + LLM − pembanding)'); ax.set_title(f'{title}: Δ dan CI 95%')
axes[0].legend(fontsize=8, frameon=False)
plt.tight_layout(); fig.savefig(FIG / 'forest_plot_vs_soft_llm.png', dpi=200, bbox_inches='tight'); plt.close(fig)

## Kasus yang dirujuk ke LLM: metrik, transisi, uji (revision/02)

In [ ]:
fl, d_fl = E.subset_metrics(df, df.flagged.values, f'ditandai ({int(df.flagged.sum())})')
au, d_au = E.subset_metrics(df, ~df.flagged.values, f'auto-accept ({int((~df.flagged).sum())})', seed=C.STAT_SEED + 1)
al, d_al = E.subset_metrics(df, np.ones(n, bool), f'semua ({n})', seed=C.STAT_SEED + 2)
SAVE(pd.concat([fl, au, al]), 'T_kasus_ditandai_metrik', 'Metrik per subset (CI 95% cluster bootstrap)')
SAVE(pd.DataFrame([d_fl, d_au, d_al]), 'T_kasus_ditandai_delta', 'Sesudah − sebelum arbitrase (Soft Voting + LLM − Weighted Soft Voting)')
tr = E.transitions(df); SAVE(tr, 'T_transisi', 'Transisi soft vote → label akhir', index=True)
rev = df[df.decision == 'revise']
dirs = (rev.assign(dari=rev['Weighted Soft Voting'].map(C.LABEL_MAP_INV), ke=rev[E.REF].map(C.LABEL_MAP_INV),
                   benar_sesudah=rev[E.REF] == rev.y_true, benar_sebelum=rev['Weighted Soft Voting'] == rev.y_true)
        .groupby(['dari', 'ke']).agg(n=('y_true', 'size'), benar_sesudah=('benar_sesudah', 'sum'), benar_sebelum=('benar_sebelum', 'sum'))
        .reset_index().sort_values('n', ascending=False))
SAVE(dirs, 'T_arah_revisi', 'Arah revisi LLM')
ft = E.flagged_tests(df); SAVE(ft, 'T_kasus_ditandai_uji', 'Uji sesudah vs sebelum arbitrase pada kasus ditandai')
tr

## Perbandingan dengan LLM-only dan LLM-always; panggilan/token/waktu API

In [ ]:
subsets = [(f'semua {n}', np.ones(n, bool)), (f'ditandai {int(df.flagged.sum())}', df.flagged.values),
           (f'auto-accept {int((~df.flagged).sum())}', ~df.flagged.values)]
vs = E.compare_vs(df, [E.REF, 'LLM-only', 'LLM-always', 'Weighted Soft Voting', 'IndoBERT'], E.REF, subsets)
SAVE(vs, 'T_hybrid_vs_LLM', 'Δ = pembanding − Soft Voting + LLM; CI 95% cluster bootstrap; p permutasi cluster (tanpa koreksi)')
ao = E.compare_vs(df, ['LLM-always', 'LLM-only'], 'LLM-only', subsets, seed=C.STAT_SEED + 300)
SAVE(ao, 'T_always_vs_only', 'Δ = LLM-always − LLM-only')
usage = pd.read_csv(D.llm / 'usage_summary.csv') if (D.llm / 'usage_summary.csv').exists() and (D.llm / 'usage_summary.csv').stat().st_size > 5 else pd.DataFrame()
SAVE(usage if len(usage) else pd.DataFrame([{'catatan': 'tidak ada panggilan API baru (semua dari cache)'}]), 'T_api_usage',
     'Panggilan, token, waktu, biaya API ronde 3 (hanya panggilan sungguhan)')
key = vs[vs.skema.isin(['LLM-only', 'LLM-always']) & vs.metrik.isin(['accuracy', 'asc_f1', 'overall_f1'])]
key.assign(**{'Δ [CI], p': [f'{d:+.4f} [{l:+.4f}; {h:+.4f}] p={p:.4f}' for d, l, h, p in zip(key.delta_vs_ref, key.delta_lo, key.delta_hi, key.p_permutasi)]}
           ).pivot_table(index=['subset', 'skema'], columns='metrik', values='Δ [CI], p', aggfunc='first')

## Analisis sensitivitas: LLM-only qwen3-max-2026-01-23 (cache lama) vs LLM-only qwen3.7-max-2026-06-08

Snapshot qwen3-max ditarik penyedia sehingga semua kondisi LLM dijalankan dengan `C.LLM_MODEL` (`ganti_model_llm.md`). Pembanding ini
hanya menunjukkan seberapa besar penggantian model mengubah baseline LLM-only (prompt, parameter, parsing, fallback identik); tidak
dipakai untuk memilih apa pun. Kesepakatan label = proporsi pasangan berlabel sama dan Cohen's kappa 4 kelas (CI cluster bootstrap per tweet).

In [ ]:
SENS_COL = f'LLM-only {C.LLM_MODEL_OLD}'
sl = pd.read_csv(D.llm / 'llm_only_qwen3max_test.csv'); sl['pair_id'] = sl['pair_id'].astype(str)
sl = df[['pair_id']].merge(sl, on='pair_id', how='left')
assert sl.final_label.notna().all(), 'label sensitivitas qwen3-max belum lengkap (R3_05)'
df[SENS_COL] = sl.final_label.map(C.LABEL_MAP).astype(int).values
sens = E.compare_vs(df, ['LLM-only', SENS_COL], 'LLM-only', subsets[:1], seed=C.STAT_SEED + 400)
sens['model'] = sens.skema.map({'LLM-only': C.LLM_MODEL, SENS_COL: C.LLM_MODEL_OLD})
SAVE(sens, 'T_sensitivitas_llm_only_model',
     f'LLM-only {C.LLM_MODEL_OLD} vs {C.LLM_MODEL}: metrik dan CI 95% cluster bootstrap; Δ = {C.LLM_MODEL_OLD} − {C.LLM_MODEL}, p permutasi cluster')
ids_a, Ca = ru.cluster_confusions(df, [SENS_COL], true_col='LLM-only'); Ca = Ca[:, 0].astype(float)
BCa = ru.boot_cms(ru.bootstrap_weights(len(ids_a), C.B_BOOT, C.STAT_SEED + 401), Ca)
agree = lambda cm: np.trace(cm, axis1=-2, axis2=-1) / cm.sum(axis=(-1, -2))
from sklearn.metrics import cohen_kappa_score
kappa_pt = float(ru.kappa_from_cm(Ca.sum(0)))
assert np.isclose(kappa_pt, cohen_kappa_score(df['LLM-only'], df[SENS_COL]))
agr = pd.DataFrame([{'metrik': 'kesepakatan label (%)', 'estimasi': 100 * float(agree(Ca.sum(0))), 'ci_lo': 100 * ru.pct_ci(agree(BCa))[0],
                     'ci_hi': 100 * ru.pct_ci(agree(BCa))[1]},
                    {'metrik': "Cohen's kappa 4 kelas", 'estimasi': kappa_pt, 'ci_lo': ru.pct_ci(ru.kappa_from_cm(BCa))[0],
                     'ci_hi': ru.pct_ci(ru.kappa_from_cm(BCa))[1]}]).assign(n_pasangan=n, n_tweet=len(ids_a))
SAVE(agr, 'T_sensitivitas_kesepakatan', f'Kesepakatan label LLM-only {C.LLM_MODEL_OLD} vs {C.LLM_MODEL} (CI 95% cluster bootstrap)')
SAVE(pd.crosstab(df['LLM-only'].map(C.LABEL_MAP_INV), df[SENS_COL].map(C.LABEL_MAP_INV)).rename_axis(index=C.LLM_MODEL, columns=C.LLM_MODEL_OLD),
     'T_sensitivitas_silang', 'Tabulasi silang label LLM-only kedua model', index=True, ff='.0f')
display(agr.round(4))
sens[sens.metrik.isin(['accuracy', 'asc_f1', 'overall_f1'])].pivot_table(index='model', columns='metrik', values='nilai').round(4)

## Tabel 16 (ablasi preprocessing V1–V6) dan Tabel 17 (skema ensemble terpilih di dev)

In [ ]:
rows = []
for nm in ['IndoBERT', 'XLM-RoBERTa', 'mBERT']:
    for v in C.TEXT_VERSIONS_ABLATION:
        m = ru.sklearn_metrics(df.y_true.values, df[f'{nm} {v.upper()}'].values)
        rows.append({'model': nm, 'versi': v.upper(), 'ACD F1': m['acd_f1'], 'ASC F1': m['asc_f1'], 'Overall F1': m['overall_f1'], 'Akurasi': m['accuracy']})
t16 = pd.DataFrame(rows)
if frozen.get('ablation_dev_file') and os.path.exists(frozen['ablation_dev_file']):
    dv = pd.read_csv(frozen['ablation_dev_file']).assign(model=lambda d: d.model.map({'indobert': 'IndoBERT', 'xlm_roberta': 'XLM-RoBERTa', 'mbert': 'mBERT'}))
    t16 = t16.merge(dv[['model', 'versi', 'Overall_dev']], on=['model', 'versi'], how='left')
SAVE(t16, 'T16_ablasi_preprocessing', 'Tabel 16 — ablasi V1–V6 (inferensi saja; model dilatih V1). Tidak dipakai memilih apa pun.')
t17cols = ['Tabel 17 terpilih', 'Weighted Soft Voting', 'Hard Voting', 'IndoBERT']
t17, _ = E.metrics_ci(df, t17cols)
t17['skema_terpilih_dev'] = frozen['table17']['selected_key']; t17['ASC_dev_terpilih'] = frozen['table17']['dev_metrics']['asc']
SAVE(t17, 'T17_skema_ensemble_terpilih', 'Tabel 17 — skema dipilih dengan ASC F1 dev (grid NB05B), dilaporkan di uji')
t16.pivot(index='versi', columns='model', values='Overall F1').round(4)

## Tabel 6: audit silver dengan konfigurasi baru (score_sentence, θ = 15,0)

In [ ]:
ns = L.base_namespace(LEXICON_DIR=os.path.join(C.RESOURCES, 'inset_lexicon'), KEYWORDS_PATH=os.path.join(C.RESOURCES, 'aspect_keywords.json'))
L.exec_legacy(['nb02_load_lexicon', 'nb02_scorer', 'nb02_score_windowed', 'nb02_detect_aspect'], ns)
lab_src = L.legacy_source('nb02_labeling_loop')
tw = pd.read_csv(C.TWEETS_PREPROCESSED, dtype={'tweet_id': str})
# Label gold diambil dari file pasangan (dev; uji hanya di mode nyata) — DRY_RUN tidak memuat label uji sama sekali
gold_sp = {'dev': pd.read_csv(C.DEV_PAIRS, dtype={'tweet_id': str})}
if not DRY_RUN:
    gold_sp['uji'] = pd.read_csv(C.TEST_PAIRS, dtype={'tweet_id': str})
for g in gold_sp.values():
    g['aspek'] = g['pair_id'].astype(str).map(lambda p: next(a for a in C.ASPECTS if p.endswith('_' + a)))
loop_ns = dict(ns, df_for_silver=tw[tw.tweet_id.isin(set().union(*[set(g.tweet_id) for g in gold_sp.values()]))].copy(),
               BEST_THETA=C.SILVER_THETA, score_windowed=lambda text, aspect, window=5: ns['score_sentence'](text))
exec(compile(lab_src, '<legacy:nb02_labeling_loop>', 'exec'), loop_ns)
rows = []
for sp, g in gold_sp.items():
    d = g.merge(loop_ns['df_silver'][['tweet_id', 'aspek', 'label_silver']], on=['tweet_id', 'aspek'], how='left')
    assert len(d) == len(g) and d.label_silver.notna().all()
    d['y_true'], d['silver'] = d.label.astype(int), d.label_silver.map(C.LABEL_MAP)
    idx, Cm = ru.cluster_confusions(d, ['silver']); Cm = Cm[:, 0].astype(float)
    BC = ru.boot_cms(ru.bootstrap_weights(len(idx), C.B_BOOT, C.STAT_SEED), Cm)
    for nm, f in [('akurasi', lambda cm: ru.metrics_from_cm(cm)['accuracy']), ('macro-F1 4 kelas', lambda cm: ru.metrics_from_cm(cm)['overall_f1']),
                  ('kappa 4 kelas', ru.kappa_from_cm), ('macro-F1 3 kelas sentimen (ASC)', lambda cm: ru.metrics_from_cm(cm)['asc_f1']),
                  ('ACD F1', lambda cm: ru.metrics_from_cm(cm)['acd_f1'])]:
        lo, hi = ru.pct_ci(f(BC)); rows.append({'split': sp, 'n_pasangan': len(d), 'metrik': nm, 'estimasi': float(f(Cm.sum(0))), 'ci_lo': lo, 'ci_hi': hi})
t6 = pd.DataFrame(rows)
SAVE(t6, 'T6_audit_silver_baru', 'Tabel 6 — label silver (score_sentence, θ=15,0) vs gold; θ dipilih hanya dengan dev (dev in-sample untuk θ)')
t6.assign(nilai=[E.fmt_ci(e, l, h, 3) for e, l, h in zip(t6.estimasi, t6.ci_lo, t6.ci_hi)]).pivot(index='metrik', columns='split', values='nilai')

## Disagreement (dev dari R3_03; target untuk deskripsi) dan Gambar 2–4

In [ ]:
dis_dev = pd.read_csv(C.DEV_DIR / 'disagreement_dev.csv', index_col=0)
nice = {'indobert': 'IndoBERT', 'xlm_roberta': 'XLM-RoBERTa', 'mbert': 'mBERT'}
dis_t = pd.DataFrame({f'{nice[a]} vs {nice[b]}': [100 * (df[nice[a]] != df[nice[b]]).mean()] for a, b in itertools.combinations(C.MODELS, 2)},
                     index=['target']).T
dis_t['dev (R3_03)'] = [dis_dev.loc[a, b] for a, b in itertools.combinations(C.MODELS, 2)]
SAVE(dis_t, 'T2_disagreement', 'Disagreement antar model (%): dev (dipakai untuk analisis) dan target (deskriptif)', index=True, ff='.1f')

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(dis_dev.rename(index=nice, columns=nice), annot=True, fmt='.1f', cmap='Purples', ax=ax, cbar_kws={'label': '%'})
ax.set_title('Gambar 2 — Disagreement antar model (dev, %)'); plt.tight_layout()
fig.savefig(FIG / 'gambar2_disagreement_dev.png', dpi=200, bbox_inches='tight'); plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(16, 5.9))
pie_kw = dict(startangle=90, counterclock=False, autopct='%1.1f%%', wedgeprops={'edgecolor': 'white', 'linewidth': 2},
              textprops={'fontsize': 12, 'fontweight': 'bold'})
n_split, n_unan = int((df.vote_type == 'terpecah').sum()), int((df.vote_type == 'bulat').sum())
axes[0].pie([n_split, n_unan], labels=[f'Terpecah\n({n_split})', f'Bulat\n({n_unan})'], colors=['#FFC107', '#4CAF50'], **pie_kw)
axes[0].set_title('Proporsi Kesepakatan Model\n(Vote Type)', fontsize=14, fontweight='bold')
n_llm = int(df.flagged.sum())
axes[1].pie([n_llm, n - n_llm], labels=[f'Needs LLM\n({n_llm})', f'Auto-Accept\n({n - n_llm})'], colors=['#F44336', '#2196F3'], **pie_kw)
axes[1].set_title('Proporsi Routing Evaluasi\n(Kebutuhan LLM Judge)', fontsize=14, fontweight='bold')
mg = df.soft_vote_margin
sns.histplot(mg, bins=25, kde=True, color='#9C27B0', edgecolor='white', alpha=0.5, ax=axes[2])
axes[2].axvline(C.MARGIN_THRESHOLD, color='red', linestyle='--', linewidth=2, label='Threshold Ambigu (0.15)')
axes[2].text(0.05, 0.95, f'Mean: {mg.mean():.2f}\nMedian: {mg.median():.3f}\nMin: {mg.min():.3f}', transform=axes[2].transAxes,
             fontsize=12, va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
axes[2].set_title('Distribusi Margin Keyakinan\n(Weighted Soft Voting)', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Soft Vote Margin (0.0 - 1.0)'); axes[2].set_ylabel('Jumlah Pasangan (Pairs)'); axes[2].legend(loc='upper center')
plt.tight_layout(); fig.savefig(FIG / 'gambar3_ensemble_stats_plots.png', dpi=200, bbox_inches='tight'); plt.close(fig)

lbl = {'IndoBERT': 'IndoBERT', 'XLM-RoBERTa': 'XLM-RoBERTa', 'mBERT': 'mBERT', 'Hard Voting': 'Hard Voting',
       'Weighted Soft Voting': 'Soft Voting', 'Soft Voting + LLM': 'Soft Voting +\nLLM Judge'}
fig, ax = plt.subplots(figsize=(13.9, 6.9)); x = np.arange(6); w = 0.35
b1 = ax.bar(x - w / 2, main.acd_f1, w, label='ACD Macro-F1', color='#4CAF50', edgecolor='white', linewidth=1.5)
b2 = ax.bar(x + w / 2, main.asc_f1, w, label='ASC Macro-F1', color='#FF9800', edgecolor='white', linewidth=1.5)
for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01, f'{b.get_height():.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold', color='#333')
ax.set_xticks(x); ax.set_xticklabels([lbl[s] for s in main.skema], fontsize=12, fontweight='bold'); ax.set_ylim(0, 1.0)
ax.set_ylabel('Macro-F1 Score', fontsize=14, fontweight='bold'); ax.set_title('Perbandingan ACD F1 dan ASC F1 pada Enam Skema Prediksi', fontsize=16, fontweight='bold')
ax.yaxis.grid(True, linestyle='--', alpha=0.7); ax.set_axisbelow(True); ax.legend(loc='upper left', fontsize=12)
plt.tight_layout(); fig.savefig(FIG / 'gambar4_scheme_comparison_plot.png', dpi=200, bbox_inches='tight'); plt.close(fig)
print('Gambar disimpan:', sorted(p.name for p in FIG.glob('*.png')))

## perbandingan_lama_vs_baru.md

In [ ]:
OLD = {  # angka naskah lama (run θ=3,0, alias qwen3-max) dan revisi sebelumnya; sumber dicatat
    'main': {s: dict(zip(E.METRICS, v)) for s, v in ru.PAPER_NUMBERS.items()},
    'retain_revise': (41, 53), 'n_flagged': 94,
    'mcnemar_holm': {'IndoBERT': 0.021, 'Hard Voting': 0.001, 'Weighted Soft Voting': 0.0002, 'XLM-RoBERTa': 0.001, 'mBERT': 0.000008},
    'asc_holm_revisi': {'IndoBERT': 1.000, 'Hard Voting': 0.168, 'Weighted Soft Voting': 0.105, 'XLM-RoBERTa': 0.347, 'mBERT': 0.014},
    'table17': ('06_SoftVote_weighted_indobert_mbert (dipilih dengan uji)', 0.4039),
    'ablation_overall': {('IndoBERT', 'V1'): 0.4866, ('IndoBERT', 'V2'): 0.4863, ('IndoBERT', 'V3'): 0.4877, ('IndoBERT', 'V4'): 0.4864,
                         ('IndoBERT', 'V5'): 0.4745, ('IndoBERT', 'V6'): 0.4841, ('XLM-RoBERTa', 'V1'): 0.4729, ('XLM-RoBERTa', 'V2'): 0.4650,
                         ('XLM-RoBERTa', 'V3'): 0.4721, ('XLM-RoBERTa', 'V4'): 0.4767, ('XLM-RoBERTa', 'V5'): 0.4824, ('XLM-RoBERTa', 'V6'): 0.4693,
                         ('mBERT', 'V1'): 0.4670, ('mBERT', 'V2'): 0.4670, ('mBERT', 'V3'): 0.4813, ('mBERT', 'V4'): 0.4621,
                         ('mBERT', 'V5'): 0.4838, ('mBERT', 'V6'): 0.4778},
    'disagreement_test': {'IndoBERT vs XLM-RoBERTa': 12.0, 'IndoBERT vs mBERT': 12.0, 'XLM-RoBERTa vs mBERT': 10.2},
    'silver_audit_test': {'akurasi': 0.691, 'macro-F1 4 kelas': 0.477, 'kappa 4 kelas': 0.437, 'macro-F1 3 kelas sentimen (ASC)': 0.408, 'ACD F1': 0.836},
    'llm_only': {'accuracy': 0.800, 'asc_f1': 0.523, 'overall_f1': 0.604}, 'llm_always': {'accuracy': 0.811, 'asc_f1': 0.576, 'overall_f1': 0.643},
}
f4 = lambda x: '—' if x is None or (isinstance(x, float) and np.isnan(x)) else (f'{x:.4f}' if isinstance(x, float) else str(x))
R = []
add = lambda grp, item, old, new: R.append(f'| {grp} | {item} | {f4(old)} | {f4(new)} | ' +
                                           (f'{new - old:+.4f}' if isinstance(old, float) and isinstance(new, float) else '—') + ' |')
mm = main.set_index('skema')
for s in E.SCHEMES6:
    for k in E.METRICS:
        add('Tabel utama', f'{s} — {E.METRIC_LABEL[k]}', OLD['main'][s][k], float(mm.loc[s, k]))
add('LLM', 'Kasus dirujuk ke LLM', OLD['n_flagged'], int(df.flagged.sum()))
add('LLM', 'Retain / revise', f'{OLD["retain_revise"][0]} / {OLD["retain_revise"][1]}',
    f"{int((df.decision == 'retain').sum())} / {int((df.decision == 'revise').sum())}")
for o in ['IndoBERT', 'Hard Voting', 'Weighted Soft Voting', 'XLM-RoBERTa', 'mBERT']:
    r = mc[((mc.skema_A == E.REF) & (mc.skema_B == o)) | ((mc.skema_B == E.REF) & (mc.skema_A == o))].iloc[0]
    add('Tabel 15', f'McNemar naskah p Holm vs {o}', OLD['mcnemar_holm'][o], float(r.p_holm_naskah))
    r = pw_asc[(pw_asc.skema_A == E.REF) & (pw_asc.skema_B == o)].iloc[0]
    add('Tabel 15', f'ΔASC permutasi cluster p Holm vs {o} (revisi)', OLD['asc_holm_revisi'][o], float(r.p_holm))
t17m = t17.set_index('skema').loc['Tabel 17 terpilih']
add('Tabel 17', 'Skema terpilih', OLD['table17'][0], f"{frozen['table17']['selected_key']} (dipilih dengan dev)")
add('Tabel 17', 'ASC F1 skema terpilih (uji)', OLD['table17'][1], float(t17m.asc_f1))
for (mdl, v), old in OLD['ablation_overall'].items():
    add('Tabel 16', f'{mdl} {v} — Overall F1', old, float(t16[(t16.model == mdl) & (t16.versi == v)]['Overall F1'].iloc[0]))
for kname, old in OLD['disagreement_test'].items():
    add('Tabel 2', f'Disagreement {kname} (%, uji)', old, float(dis_t.loc[kname, 'target']))
if 'uji' in t6.split.values:
    for kname, old in OLD['silver_audit_test'].items():
        add('Tabel 6', f'Silver vs gold uji — {kname}', old, float(t6[(t6.split == 'uji') & (t6.metrik == kname)].estimasi.iloc[0]))
for nm, key2 in [('LLM-only', 'llm_only'), ('LLM-always', 'llm_always')]:
    for k in ['accuracy', 'asc_f1', 'overall_f1']:
        add('Baseline LLM', f'{nm} — {E.METRIC_LABEL[k]}', OLD[key2][k],
            float(vs[(vs.subset == subsets[0][0]) & (vs.skema == nm) & (vs.metrik == k)].nilai.iloc[0]))
md = ('# Perbandingan angka naskah lama vs latih ulang ronde 3\n\n'
      f"Mode: **{'DRY_RUN — label DEV, BUKAN hasil uji' if DRY_RUN else 'NYATA — data uji'}**. Lama = naskah (θ = 3,0, alias qwen3-max) "
      f'kecuali ditandai "revisi" (analisis revisi sebelumnya). Baru = ronde 3 (score_sentence, θ = 15,0, snapshot {C.LLM_MODEL}; '
      f'{C.LLM_MODEL_OLD} ditarik penyedia — lihat ganti_model_llm.md dan T_sensitivitas_*).\n\n'
      '| Kelompok | Angka | Lama | Baru | Selisih |\n|---|---|---|---|---|\n' + '\n'.join(R) + '\n')
(OUT / 'perbandingan_lama_vs_baru.md').write_text(md, encoding='utf-8')
if not DRY_RUN:
    C.assert_writable(C.OUT_R3 / 'perbandingan_lama_vs_baru.md').write_text(md, encoding='utf-8')
print(md[:1500])

In [ ]:
mapping = """# Pemetaan tabel/gambar naskah → file (PERLU DIKONFIRMASI PENELITI)

Penomoran naskah tidak tersedia di repo; pemetaan ini berdasarkan isi. File ada di folder evaluasi (CSV + Markdown) dan `figures/`.

| Naskah (perkiraan) | Isi | File |
|---|---|---|
| Tabel 2 | Disagreement antar model | `T2_disagreement` (dev = untuk analisis; target = deskriptif) |
| Tabel 5 | Enam skema: ACD/ASC/Overall F1, akurasi (+CI) | `T_utama_enam_skema`, `T_utama_enam_skema_ci` |
| Tabel 6 | Audit label silver konfigurasi baru vs gold | `T6_audit_silver_baru` |
| Tabel 8–14 | Per kelas, per aspek, kasus ditandai, transisi, arah revisi, LLM-only/always, API | `T_per_kelas_*`, `T_per_aspek`, `T_kasus_ditandai_*`, `T_transisi`, `T_arah_revisi`, `T_hybrid_vs_LLM`, `T_always_vs_only`, `T_api_usage` |
| (baru, lampiran) | Sensitivitas LLM-only: qwen3-max-2026-01-23 vs qwen3.7-max-2026-06-08 | `T_sensitivitas_llm_only_model`, `T_sensitivitas_kesepakatan`, `T_sensitivitas_silang` |
| Tabel 15 | Uji signifikansi 15 perbandingan, ditandai ✔/✘ setelah Holm | `T15_signifikansi` (+ `T_berpasangan_*`, `T_mcnemar`) |
| Tabel 16 | Ablasi preprocessing V1–V6 | `T16_ablasi_preprocessing` |
| Tabel 17 | Skema ensemble terpilih di dev | `T17_skema_ensemble_terpilih` |
| Gambar 2 | Heatmap disagreement (dev) | `figures/gambar2_disagreement_dev.png` |
| Gambar 3 | Vote type, routing, margin | `figures/gambar3_ensemble_stats_plots.png` |
| Gambar 4 | ACD/ASC F1 enam skema | `figures/gambar4_scheme_comparison_plot.png` |
"""
(OUT / 'pemetaan_tabel_naskah.md').write_text(mapping, encoding='utf-8')
L.write_json(OUT / 'DONE.json', {'finished_at': L.now_utc(), 'dry_run': DRY_RUN, 'n_pairs': n,
                                 'files': sorted(p.name for p in OUT.rglob('*') if p.is_file())})
print('✓ Evaluasi selesai:', OUT)